# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [2]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


In [5]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [6]:
# Use esta célula para criar sua análise exploratória.

colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()

,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


In [7]:
# Preencha com uma variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

variavel_x = "taxa_abandono_carrinho_pct"

if variavel_x not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x,
    y="taxa_conversao_pct",
    title="Relação com a taxa de conversão",
)
fig.show()

In [14]:
# Preencha com uma segunda variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

variavel_x2 = "profundidade_scroll_pct"

if variavel_x2 not in features:
    raise ValueError("Preencha variavel_x2 com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x2,
    y="taxa_conversao_pct",
    title="Relação com a taxa de conversão",
)
fig.show()

In [26]:
import plotly.express as px

corr = df[
    [
        "taxa_abandono_carrinho_pct",
        "profundidade_scroll_pct",
        "tempo_primeiro_clique_s",
        "taxa_conversao_pct",
    ]
].corr()

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Matriz de correlação entre as variáveis"
)

fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Para a primeira análise de sensibilidade, considerei a taxa de abandono do carrinho e a taxa de conversão (que já foi proposta no próprio exercício). Observando a matriz, a taxa de abandono apresenta uma relação negativa relativamente forte com a taxa de conversão (-0,643), ou seja, quanto maior o abandono do carrinho, menor tende a ser a conversão.

Depois, no segundo gráfico, analisei a profundidade de scroll (a segunda relação mais forte, com 0,485) e a taxa de conversão. Tem uma tendência positiva, indicando que usuários que navegam mais pelo aplicativo tendem a apresentar maiores taxas de conversão. Escolhi essas variáveis já que as duas mostraram ter influência sobre a taxa de conversão - como observado no heatmap.

## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [15]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
]
target = "taxa_conversao_pct"

X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

,métrica,valor
0,MAE,0.317
1,RMSE,0.402


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O modelo apresentou um MAE (erro absoluto médio) de aproximadamente 0,317 e um RMSE (raiz do erro quadrático médio) de 0,402. Essas métricas indicam o quanto as previsões do modelo se desviam dos valores reais, sendo que o RMSE penaliza mais erros maiores. Como os valores obtidos são relativamente baixos, podemos dizer que o modelo consegue representar de forma razoável a relação entre a taxa de abandono do carrinho, a profundidade de scroll e a taxa de conversão, com previsões próximas aos valores observados.

## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [16]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032},
 5.868747841831929)

In [17]:
# Preencha com duas variáveis escolhidas na Parte 1.
# Use exatamente os nomes que aparecem em features.

variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct"]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.598,-4.612,-0.461
1,profundidade_scroll_pct,62.449,68.694,5.869,6.015,2.499,0.250


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

Para essa análise de sensibilidade, mantive as variáveis taxa_abandono_carrinho_pct e profundidade_scroll_pct. Foi feita uma variação de 10% em cada uma delas para avaliar o impacto na taxa de conversão. Aumentando a taxa de abandono do carrinho em 10%, a taxa de conversão prevista caiu cerca de 4,6%, resultando em um índice de sensibilidade de -0,461. Já um aumento de 10% na profundidade de scroll gerou um crescimento de aproximadamente 2,5% na taxa de conversão, com índice de sensibilidade de 0,250. Isso mostra que a taxa de abandono do carrinho tem um impacto maior sobre a conversão e, por isso, é a variável mais sensível entre as duas que eu escolhi.

## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

Com base na minha análise, a variável que deveria ser priorizada é a taxa de abandono do carrinho. Tendo uma correlação relativamente forte com a taxa de conversão, ela também foi a variável mais sensível na análise. O aumento de 10% na taxa de abandono provocou uma queda de aproximadamente 4,6% na taxa de conversão, mostrando que pequenas mudanças nessa variável têm um impacto significativo no resultado final.

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

Na minha opinião, uma ação possível seria otimizar o processo de checkout para ser mais simples e intuitivo. Reduzir a quantidade de etapas, facilitar as opções de pagamento e diminuir possíveis pontos de fricção podem ajudar a reduzir o abandono do carrinho. Como essa variável possui grande influência sobre a taxa de conversão, melhorias nessa etapa podem gerar ganhos relevantes no desempenho desse aplicativo.

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [22]:
# Use esta célula para sua simulação.

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

,0
count,1000.000
mean,5.873
std,0.349
min,4.545
10%,5.429
25%,5.631
50%,5.872
75%,6.101
90%,6.324
max,6.935


In [23]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

A simulação de Monte Carlo mostrou que a taxa de conversão prevista possui média de aproximadamente 5,87% e desvio padrão de cerca de 0,35. Ainda, 80% das simulações ficaram entre aproximadamente 5,43% e 6,32%, mostrando que a maior parte dos cenários produz resultados relativamente próximos entre si. Isso indica que, mesmo considerando a incerteza nas variáveis de entrada, a taxa de conversão tende a permanecer em uma faixa estável. Então, priorizar a redução da taxa de abandono do carrinho apresenta um risco relativamente baixo, já que pequenas variações nas condições do sistema não provocam mudanças drásticas nas previsões.

## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.